# Running this experiment

See README.md for setup, required inputs, and limitations. Run cells in order from a fresh kernel. Saved outputs have been cleared; historical metrics are not newly verified results. Environment installation and external training commands are manual setup steps.

In [ ]:
from project_utils import project_path

# Override these paths for your local datasets. PROJECT_DATA_DIR changes their base.
INPUT_PATHS = {
    "ARCHIVE_ZIP": project_path("archive.zip"),
    "TRAINING": project_path("Training"),
    "TESTING": project_path("Testing"),
}

In [ ]:
# Check GPU
pass  # Environment setup moved to README; use local inputs.

In [ ]:
# Get helper functions file from Daniel Bourke's Github (He's an absolute Legend)
pass  # Environment setup moved to README; use local inputs.

In [ ]:
# Import series of helper functions for the notebook
from helper_functions import (
    plot_loss_curves,
    unzip_data,
    walk_through_dir,
)

In [ ]:
pass  # Environment setup moved to README; use local inputs.
pass  # Environment setup moved to README; use local inputs.

# Data Preprocessing

In [ ]:
# Download and unzip data
unzip_data(INPUT_PATHS["ARCHIVE_ZIP"])

In [ ]:
# How many images/classes are there in training?
walk_through_dir(INPUT_PATHS["TRAINING"])

In [ ]:
# How many images/classes are there in testing?
walk_through_dir(INPUT_PATHS["TESTING"])

In [ ]:
# Set training and testing directories
train_dir = INPUT_PATHS["TRAINING"]
test_dir = INPUT_PATHS["TESTING"]

In [ ]:
# Setup data inputs
import tensorflow as tf

IMG_SIZE = (224, 224)
train_data = tf.keras.preprocessing.image_dataset_from_directory(
    train_dir, label_mode="categorical", image_size=IMG_SIZE
)

test_data = tf.keras.preprocessing.image_dataset_from_directory(
    test_dir, label_mode="categorical", image_size=IMG_SIZE, shuffle=False
)  # don't shuffle test data for prediction analysis

In [ ]:
# Create checkpoint callback to save model for later use
checkpoint_path = "brain_tumor_classification_model_checkpoint.weights.h5"
checkpoint_callback = tf.keras.callbacks.ModelCheckpoint(
    checkpoint_path,
    save_weights_only=True,  # save only the model weights
    monitor="val_accuracy",  # save the model weights which score the best validation accuracy
    save_best_only=True,
)  # only keep the best model weights on file (delete the rest)

In [ ]:
# Import the required modules for model creation
from tensorflow.keras import layers

# Setup data augmentation
from experiment_comparison_notebook_efficientnetb0_mobilenetv3small import (
    build_data_augmentation_cell_13,
)

data_augmentation = build_data_augmentation_cell_13()

# EfficientNetB0 Model (Fully Trainable)

In [ ]:
# Setup base model
base_model = tf.keras.applications.EfficientNetB0(include_top=False)
base_model.trainable = False

# Setup model architecture
inputs = layers.Input(shape=(224, 224, 3), name="input_layer")  # shape of input image
x = data_augmentation(inputs)  # augment images (only happens during training)
x = base_model(x, training=False)
x = layers.GlobalAveragePooling2D(name="global_average_pooling")(
    x
)  # pool the outputs of the base model
outputs = layers.Dense(len(train_data.class_names), activation="softmax", name="output_layer")(
    x
)  # same number of outputs as classes
model_efficientnetb0 = tf.keras.Model(inputs, outputs)

In [ ]:
# Get a summary of our model
model_efficientnetb0.summary()

In [ ]:
# Unfreeze all of the layers in the base model
base_model.trainable = True

In [ ]:
# Compile the mdoel
model_efficientnetb0.compile(
    loss="categorical_crossentropy",
    optimizer=tf.keras.optimizers.Adam(1e-4),
    metrics=["accuracy"],
)

In [ ]:
# What layers in the model are trainable?
for layer in model_efficientnetb0.layers:
    print(layer.name, layer.trainable)

In [ ]:
# Check which layers are trainable
for layer_number, layer in enumerate(base_model.layers):
    print(layer_number, layer.name, layer.trainable)

In [ ]:
# Train for 10 epochs
epochs = 10

history_efficientnetb0 = model_efficientnetb0.fit(
    train_data,
    epochs=epochs,
    validation_data=test_data,
    validation_steps=int(len(test_data)),
)

In [ ]:
# Save model to drive so it can be used later
model_efficientnetb0.save("efficientnetb0_model.keras")

In [ ]:
# Evaluate the model
loaded_loss, loaded_accuracy = model_efficientnetb0.evaluate(test_data)
loaded_loss, loaded_accuracy

In [ ]:
# Make predictions with model
pred_probs = model_efficientnetb0.predict(
    test_data, verbose=1
)  # set verbosity to see how long it will take

In [ ]:
# What's the shape of our predictions?
pred_probs.shape

In [ ]:
# How do they look?
pred_probs[:10]

In [ ]:
# We get one prediction probability per class
print(f"Number of prediction probabilities for sample 0: {len(pred_probs[0])}")
print(f"What prediction probability sample 0 looks like:\n {pred_probs[0]}")
print(
    f"The class with the highest predicted probability by the model for sample 0: {pred_probs[0].argmax()}"
)

In [ ]:
# Get the class predicitons of each label
pred_classes = pred_probs.argmax(axis=1)

# How do they look?
pred_classes[:100]

In [ ]:
y_labels = []
for (
    images,
    labels,
) in test_data.unbatch():  # unbatch the test data and get images and labels
    y_labels.append(
        labels.numpy().argmax()
    )  # append the index which has the largest value (labels are one-hot)
y_labels[:10]  # check what they look like (unshuffled)

In [ ]:
# How many labels are there? (should be the same as how many prediction probabilities we have)
len(y_labels)

In [ ]:
# Get accuracy score by comparing predicted classes to ground truth labels
from sklearn.metrics import accuracy_score

sklearn_accuracy = accuracy_score(y_labels, pred_classes)
sklearn_accuracy

In [ ]:
# Does the evaluate method compare to the Scikit-Learn measured accuracy?
import numpy as np

print(
    f"Close? {np.isclose(loaded_accuracy, sklearn_accuracy)} | Difference: {loaded_accuracy - sklearn_accuracy}"
)

In [ ]:
# We'll import our make_confusion_matrix function from https://github.com/mrdbourke/tensorflow-deep-learning/blob/main/extras/helper_functions.py
# But if you run it out of the box, it doesn't really work for 101 classes...
# the cell below adds a little functionality to make it readable.
from helper_functions import make_confusion_matrix

In [ ]:
# Note: The following confusion matrix code is a remix of Scikit-Learn's
# plot_confusion_matrix function - https://scikit-learn.org/stable/modules/generated/sklearn.metrics.plot_confusion_matrix.html
import matplotlib.pyplot as plt
import numpy as np

# Our function needs a different name to sklearn's plot_confusion_matrix
from experiment_comparison_notebook_efficientnetb0_mobilenetv3small import (
    make_confusion_matrix,
)

In [ ]:
# Get the class names
class_names = test_data.class_names
class_names

In [ ]:
# Plot a confusion matrix with all 25250 predictions, ground truth labels and 101 classes
make_confusion_matrix(
    y_true=y_labels,
    y_pred=pred_classes,
    classes=class_names,
    figsize=(10, 10),
    text_size=15,
    norm=False,
    savefig=True,
)

In [ ]:
plot_loss_curves(history_efficientnetb0)

In [ ]:
from sklearn.metrics import classification_report

print(classification_report(y_labels, pred_classes))

In [ ]:
# Get a dictionary of the classification report
classification_report_dict = classification_report(y_labels, pred_classes, output_dict=True)
classification_report_dict

In [ ]:
# Create empty dictionary
class_f1_scores = {}
# Loop through classification report items
for k, v in classification_report_dict.items():
    if k == "accuracy":  # stop once we get to accuracy key
        break
    else:
        # Append class names and f1-scores to new dictionary
        class_f1_scores[class_names[int(k)]] = v["f1-score"]
class_f1_scores

In [ ]:
# Turn f1-scores into dataframe for visualization
import pandas as pd

f1_scores = pd.DataFrame(
    {
        "class_name": list(class_f1_scores.keys()),
        "f1-score": list(class_f1_scores.values()),
    }
).sort_values("f1-score", ascending=False)
f1_scores

In [ ]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(12, 25))
scores = ax.barh(range(len(f1_scores)), f1_scores["f1-score"].values)
ax.set_yticks(range(len(f1_scores)))
ax.set_yticklabels(list(f1_scores["class_name"]))
ax.set_xlabel("f1-score")
ax.set_title("F1-Scores for 4 Different Classes for EfficientnetB0")
ax.invert_yaxis()  # reverse the order


def autolabel(
    rects,
):  # Modified version of: https://matplotlib.org/examples/api/barchart_demo.html
    """
    Attach a text label above each bar displaying its height (it's value).
    """
    for rect in rects:
        width = rect.get_width()
        ax.text(
            1.0 * width,
            rect.get_y() + rect.get_height() / 1.5,
            f"{width:.2f}",
            ha="center",
            va="bottom",
        )


autolabel(scores)

# MobileNetV3Small (Fully Trainable)

In [ ]:
# Setup base model and freeze its layers (this will extract features)
base_model_2 = tf.keras.applications.MobileNetV3Small(include_top=False)
base_model_2.trainable = False

# Setup model architecture with trainable top layers
inputs = layers.Input(shape=(224, 224, 3), name="input_layer")  # shape of input image
x = data_augmentation(inputs)  # augment images (only happens during training)
x = base_model_2(
    x, training=False
)  # put the base model in inference mode so we can use it to extract features without updating the weights
x = layers.GlobalAveragePooling2D(name="global_average_pooling")(
    x
)  # pool the outputs of the base model
outputs = layers.Dense(len(train_data.class_names), activation="softmax", name="output_layer")(
    x
)  # same number of outputs as classes
model_mobilenetv3 = tf.keras.Model(inputs, outputs)

In [ ]:
# Get a summary of our model
model_mobilenetv3.summary()

In [ ]:
# Unfreeze all of the layers in the base model
base_model_2.trainable = True

In [ ]:
# Recompile model with lower learning rate
model_mobilenetv3.compile(
    loss="categorical_crossentropy",
    optimizer=tf.keras.optimizers.Adam(1e-4),  # 10x lower learning rate than default
    metrics=["accuracy"],
)

In [ ]:
# What layers in the model are trainable?
for layer in model_mobilenetv3.layers:
    print(layer.name, layer.trainable)

In [ ]:
# Check which layers are trainable
for layer_number, layer in enumerate(base_model_2.layers):
    print(layer_number, layer.name, layer.trainable)

In [ ]:
# Train for 10 epochs
epochs = 10

history_mobilenetv3 = model_mobilenetv3.fit(
    train_data,
    epochs=epochs,
    validation_data=test_data,
    validation_steps=int(len(test_data)),
)

In [ ]:
# Save model to drive so it can be used later
model_mobilenetv3.save("mobilenetv3_model.keras")

In [ ]:
# Evaluate the model
loaded_loss_2, loaded_accuracy_2 = model_mobilenetv3.evaluate(test_data)
loaded_loss_2, loaded_accuracy_2

In [ ]:
# Make predictions with model
pred_probs_2 = model_mobilenetv3.predict(
    test_data, verbose=1
)  # set verbosity to see how long it will take

In [ ]:
# How many predictions are there?
len(pred_probs_2)

In [ ]:
# How do they look?
pred_probs_2[:10]

In [ ]:
# We get one prediction probability per class
print(f"Number of prediction probabilities for sample 0: {len(pred_probs_2[0])}")
print(f"What prediction probability sample 0 looks like:\n {pred_probs_2[0]}")
print(
    f"The class with the highest predicted probability by the model for sample 0: {pred_probs_2[0].argmax()}"
)

In [ ]:
# Get the class predicitons of each label
pred_classes_2 = pred_probs_2.argmax(axis=1)

# How do they look?
pred_classes_2[:100]

In [ ]:
y_labels_2 = []
for (
    images,
    labels,
) in test_data.unbatch():  # unbatch the test data and get images and labels
    y_labels_2.append(
        labels.numpy().argmax()
    )  # append the index which has the largest value (labels are one-hot)
y_labels_2[:10]  # check what they look like (unshuffled)

In [ ]:
# How many labels are there? (should be the same as how many prediction probabilities we have)
len(y_labels_2)

In [ ]:
# Get accuracy score by comparing predicted classes to ground truth labels
from sklearn.metrics import accuracy_score

sklearn_accuracy_2 = accuracy_score(y_labels_2, pred_classes_2)
sklearn_accuracy

In [ ]:
# Does the evaluate method compare to the Scikit-Learn measured accuracy?
import numpy as np

print(
    f"Close? {np.isclose(loaded_accuracy_2, sklearn_accuracy_2)} | Difference: {loaded_accuracy_2 - sklearn_accuracy_2}"
)

In [ ]:
# We'll import our make_confusion_matrix function from https://github.com/mrdbourke/tensorflow-deep-learning/blob/main/extras/helper_functions.py
# But if you run it out of the box, it doesn't really work for 101 classes...
# the cell below adds a little functionality to make it readable.
from helper_functions import make_confusion_matrix

In [ ]:
# Plot a confusion matrix with all 25250 predictions, ground truth labels and 101 classes
make_confusion_matrix(
    y_true=y_labels_2,
    y_pred=pred_classes_2,
    classes=class_names,
    figsize=(10, 10),
    text_size=15,
    norm=False,
    savefig=True,
)

In [ ]:
plot_loss_curves(history_mobilenetv3)

In [ ]:
from sklearn.metrics import classification_report

print(classification_report(y_labels_2, pred_classes_2))

In [ ]:
# Get a dictionary of the classification report
classification_report_dict_2 = classification_report(y_labels_2, pred_classes_2, output_dict=True)
classification_report_dict_2

In [ ]:
# Create empty dictionary
class_f1_scores_2 = {}
# Loop through classification report items
for k, v in classification_report_dict_2.items():
    if k == "accuracy":  # stop once we get to accuracy key
        break
    else:
        # Append class names and f1-scores to new dictionary
        class_f1_scores_2[class_names[int(k)]] = v["f1-score"]
class_f1_scores_2

In [ ]:
# Turn f1-scores into dataframe for visualization
import pandas as pd

f1_scores_2 = pd.DataFrame(
    {
        "class_name": list(class_f1_scores_2.keys()),
        "f1-score": list(class_f1_scores_2.values()),
    }
).sort_values("f1-score", ascending=False)
f1_scores_2

In [ ]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(12, 25))
scores = ax.barh(range(len(f1_scores_2)), f1_scores_2["f1-score"].values)
ax.set_yticks(range(len(f1_scores_2)))
ax.set_yticklabels(list(f1_scores_2["class_name"]))
ax.set_xlabel("f1-score")
ax.set_title("F1-Scores for 4 Different Classes for MobilenetV3Small")
ax.invert_yaxis()  # reverse the order


def autolabel(
    rects,
):  # Modified version of: https://matplotlib.org/examples/api/barchart_demo.html
    """
    Attach a text label above each bar displaying its height (it's value).
    """
    for rect in rects:
        width = rect.get_width()
        ax.text(
            1.0 * width,
            rect.get_y() + rect.get_height() / 1.5,
            f"{width:.2f}",
            ha="center",
            va="bottom",
        )


autolabel(scores)